# Data Preprocessing Pipeline: Student Depression and Lifestyle Dataset

## 1. Dataset Loading via Kagglehub

This cell loads the Student Depression and Lifestyle dataset (100k records) directly from Kaggle using kagglehub library.
We display the shape and preview of the raw data.

In [16]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Download latest version
path = kagglehub.dataset_download("aldinwhyudii/student-depression-and-lifestyle-100k-data")
print("Path to dataset files:", path)

# Load the dataset
df = pd.read_csv(f"{path}/student_lifestyle_100k.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:\n{df.dtypes}")
df.head()

Path to dataset files: C:\Users\Vic\.cache\kagglehub\datasets\aldinwhyudii\student-depression-and-lifestyle-100k-data\versions\1
Dataset shape: (100000, 11)

Column names and types:
Student_ID              int64
Age                     int64
Gender                 object
Department             object
CGPA                  float64
Sleep_Duration        float64
Study_Hours           float64
Social_Media_Hours    float64
Physical_Activity       int64
Stress_Level            int64
Depression               bool
dtype: object


,Student_ID,Age,Gender,Department,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level,Depression
0,1001,22,Female,Science,3.50,7.3,3.3,3.4,114,5,False
1,1002,20,Male,Engineering,2.72,5.5,7.2,6.0,142,2,False
2,1003,20,Male,Medical,3.01,5.4,2.3,1.8,137,3,False
3,1004,21,Male,Engineering,3.63,8.1,2.0,4.6,130,3,False
4,1005,19,Male,Arts,3.14,6.8,2.6,4.3,4,6,False


## 1.1. Injection des variables socio-économiques

Cette cellule simule l'intégration des variables de cadrage encore absentes du dataset Kaggle actuel : prêt bancaire/dette, temps de transport, mode de logement et job étudiant.
Ces variables sont ajoutées avant l'encodage et la standardisation pour être traitées comme les autres features du pipeline.

In [17]:
import numpy as np


def inject_socio_economic_features(df):
    """
    Simule et injecte les variables socio-économiques du document de cadrage
    en créant des corrélations réalistes avec l'état actuel des étudiants.
    """
    print("[SIMULATION] Injection des variables socio-économiques du cadrage...")
    np.random.seed(42)
    n_samples = len(df)
    
    # 1. Temps de transport quotidien (loi log-normale centrée vers 35 mins, max ~120 mins)
    base_transport = np.random.lognormal(mean=3.4, sigma=0.4, size=n_samples)
    stress_modifier = df['Stress_Level'].values * 2.5
    transport_time = base_transport + stress_modifier
    df['Transportation_Time'] = np.clip(transport_time, 10, 120).astype(int)
    
    # Impact réaliste : Les longs trajets réduisent mécaniquement le temps de sommeil
    long_commute_mask = df['Transportation_Time'] > 60
    df.loc[long_commute_mask, 'Sleep_Duration'] = df.loc[long_commute_mask, 'Sleep_Duration'] - np.random.uniform(0.5, 1.5, size=long_commute_mask.sum())
    df['Sleep_Duration'] = np.clip(df['Sleep_Duration'], 3, 12)
    
    # 2. Présence d'une dette étudiante / Prêt bancaire (Binaire : 0 ou 1)
    proba_debt = np.where(df['Stress_Level'].values > 7, 0.55, 0.25)
    df['Student_Debt'] = np.random.binomial(1, proba_debt).astype(int)
    
    # 3. Avoir un job étudiant à temps partiel (Binaire : 0 ou 1)
    proba_job = np.where((df['Student_Debt'].values == 1) & (df['Stress_Level'].values > 5), 0.60, 0.20)
    df['Part_Time_Job'] = np.random.binomial(1, proba_job).astype(int)
    
    # Impact réaliste : Le job étudiant réduit le temps disponible pour réviser (CGPA en baisse)
    job_mask = df['Part_Time_Job'] == 1
    df.loc[job_mask, 'CGPA'] = df.loc[job_mask, 'CGPA'] - np.random.uniform(0.1, 0.4, size=job_mask.sum())
    df['CGPA'] = np.clip(df['CGPA'], 0.0, 4.0)
    
    # 4. Mode de logement (Catégoriel : 'Seul', 'Famille', 'Coloc')
    living_choices = ['Famille', 'Coloc', 'Seul']
    df['Living_Status'] = 'Famille'
    
    high_constraint = (df['Part_Time_Job'] == 1) | (df['Student_Debt'] == 1)
    df.loc[high_constraint, 'Living_Status'] = np.random.choice(living_choices, size=high_constraint.sum(), p=[0.2, 0.5, 0.3])
    df.loc[~high_constraint, 'Living_Status'] = np.random.choice(living_choices, size=(~high_constraint).sum(), p=[0.6, 0.2, 0.2])
    
    print("[SIMULATION] Variables ajoutées : 'Transportation_Time', 'Student_Debt', 'Part_Time_Job', 'Living_Status'")
    return df


df = inject_socio_economic_features(df)
df.head()

[SIMULATION] Injection des variables socio-économiques du cadrage...
[SIMULATION] Variables ajoutées : 'Transportation_Time', 'Student_Debt', 'Part_Time_Job', 'Living_Status'


,Student_ID,Age,Gender,Department,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level,Depression,Transportation_Time,Student_Debt,Part_Time_Job,Living_Status
0,1001,22,Female,Science,3.26427,7.30000,3.3,3.4,114,5,False,49,0,1,Seul
1,1002,20,Male,Engineering,2.72000,5.50000,7.2,6.0,142,2,False,33,0,0,Famille
2,1003,20,Male,Medical,3.01000,5.40000,2.3,1.8,137,3,False,46,1,0,Famille
3,1004,21,Male,Engineering,3.63000,7.30493,2.0,4.6,130,3,False,62,1,0,Seul
4,1005,19,Male,Arts,3.14000,6.80000,2.6,4.3,4,6,False,42,0,0,Famille


### Dataset Schema Overview

| Column Name | Description | Data Type | Range/Values |
|---|---|---|---|
| **Student_ID** | Unique identifier for each student | Integer | Unique IDs |
| **Age** | Age of the student | Integer | 18-24 |
| **Gender** | Gender of the student | String | Male, Female |
| **Department** | Field of study | String | Engineering, Business, Arts, etc. |
| **CGPA** | Cumulative Grade Point Average | Float | 0.0 - 4.0 |
| **Sleep_Duration** | Average hours of sleep per night | Float | Continuous |
| **Study_Hours** | Average hours spent studying per day | Float | Continuous |
| **Social_Media_Hours** | Average hours spent on social media per day | Float | Continuous |
| **Physical_Activity** | Average minutes of physical activity per week | Integer | Continuous |
| **Stress_Level** | Self-reported stress level | Integer | 0-10 |
| **Depression** | Mental health status | Boolean | True (Depression), False (Healthy) |
| **Transportation_Time*µ | Time from house to school | Integer |0-.. |
| **Student_debt** | Does the student have a debt | Integer | 1 (Debt), 0 (No debt) |
| **Part_Time_Job** | Does the student have a partial Job | Integer | 1 (Job), 0 (No Job) |
| **Living_status** | How does the student live| Strng | Alone, Family |

In [18]:
# Display dataset information and validation
print("=" * 70)
print("DATASET INFORMATION & VALIDATION")
print("=" * 70)

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Dataset Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
print(df.describe())

print("\n--- Depression Column Distribution ---")
print(f"Depression values: {df['Depression'].unique()}")
print(f"Unique values count: {df['Depression'].nunique()}")

DATASET INFORMATION & VALIDATION

--- Data Types ---
Student_ID               int64
Age                      int64
Gender                  object
Department              object
CGPA                   float64
Sleep_Duration         float64
Study_Hours            float64
Social_Media_Hours     float64
Physical_Activity        int64
Stress_Level             int64
Depression                bool
Transportation_Time      int64
Student_Debt             int64
Part_Time_Job            int64
Living_Status           object
dtype: object

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Student_ID           100000 non-null  int64  
 1   Age                  100000 non-null  int64  
 2   Gender               100000 non-null  object 
 3   Department           100000 non-null  object 
 4   CGPA                 1000

## 2. Data Quality Control

Check for missing values and remove non-essential identifiers (Student_ID).
This ensures data integrity before feature engineering.

In [19]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")

# Drop Student_ID column (non-informative identifier)
if 'Student_ID' in df.columns:
    df = df.drop(columns=['Student_ID'])
    print("\n[REMOVED] Student_ID column dropped.")

print(f"\nDataset shape after cleaning: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")

Missing values per column:
Student_ID             0
Age                    0
Gender                 0
Department             0
CGPA                   0
Sleep_Duration         0
Study_Hours            0
Social_Media_Hours     0
Physical_Activity      0
Stress_Level           0
Depression             0
Transportation_Time    0
Student_Debt           0
Part_Time_Job          0
Living_Status          0
dtype: int64

Total missing values: 0

[REMOVED] Student_ID column dropped.

Dataset shape after cleaning: (100000, 14)
Remaining columns: ['Age', 'Gender', 'Department', 'CGPA', 'Sleep_Duration', 'Study_Hours', 'Social_Media_Hours', 'Physical_Activity', 'Stress_Level', 'Depression', 'Transportation_Time', 'Student_Debt', 'Part_Time_Job', 'Living_Status']


## 3. Categorical Feature Encoding

Apply One-Hot Encoding to categorical features (Gender, Department, Living_Status) with drop='first'
to avoid multicollinearity and reduce dimensionality.
The socio-economic housing variable is now included in the categorical feature set.

In [20]:
# One-Hot encode categorical columns
categorical_cols = ['Gender', 'Department', 'Living_Status']

encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_features = encoder.fit_transform(df[categorical_cols])

# Get feature names
feature_names = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_features, columns=feature_names, index=df.index)

# Replace original categorical columns with encoded ones
df = df.drop(columns=categorical_cols)
df = pd.concat([df, encoded_df], axis=1)

print(f"Encoded feature names: {list(feature_names)}")
print(f"Dataset shape after encoding: {df.shape}")
print(f"\nDataset preview after One-Hot Encoding:")
df.head()

Encoded feature names: ['Gender_Male', 'Department_Business', 'Department_Engineering', 'Department_Medical', 'Department_Science', 'Living_Status_Famille', 'Living_Status_Seul']
Dataset shape after encoding: (100000, 18)

Dataset preview after One-Hot Encoding:


,Age,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level,Depression,Transportation_Time,Student_Debt,Part_Time_Job,Gender_Male,Department_Business,Department_Engineering,Department_Medical,Department_Science,Living_Status_Famille,Living_Status_Seul
0,22,3.26427,7.30000,3.3,3.4,114,5,False,49,0,1,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,20,2.72000,5.50000,7.2,6.0,142,2,False,33,0,0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,20,3.01000,5.40000,2.3,1.8,137,3,False,46,1,0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,21,3.63000,7.30493,2.0,4.6,130,3,False,62,1,0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,19,3.14000,6.80000,2.6,4.3,4,6,False,42,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## 4. Numeric Feature Standardization

Apply StandardScaler to numeric features to ensure:
- Mean = 0, Std = 1 for each feature
- Faster convergence in SGD-based algorithms
- Prevention of weight divergence due to feature scale differences
- Inclusion of the new `Transportation_Time` feature, which is continuous and must be normalized before SGD

In [21]:
# Select numeric columns for standardization
numeric_cols = [
    'Age', 'CGPA', 'Sleep_Duration', 'Study_Hours',
    'Social_Media_Hours', 'Physical_Activity', 'Stress_Level',
    'Transportation_Time'
]

# Apply StandardScaler
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("Standardization completed.")
print(f"\nFeature statistics (should be close to mean=0, std=1):")
print(df[numeric_cols].describe().round(2))

Standardization completed.

Feature statistics (should be close to mean=0, std=1):
            Age       CGPA  Sleep_Duration  Study_Hours  Social_Media_Hours  \
count  100000.0  100000.00       100000.00    100000.00           100000.00   
mean        0.0       0.00            0.00         0.00                0.00   
std         1.0       1.00            1.00         1.00                1.00   
min        -1.5      -2.94           -2.54        -2.28               -2.36   
25%        -1.0      -0.83           -0.65        -0.66               -0.67   
50%        -0.0      -0.01            0.00        -0.00               -0.00   
75%         1.0       0.82            0.65         0.65                0.67   
max         1.5       2.13            3.32         4.20                4.37   

       Physical_Activity  Stress_Level  Transportation_Time  
count          100000.00     100000.00            100000.00  
mean                0.00         -0.00                 0.00  
std                

## 5. Target Variable Transformation

Convert Depression column from Boolean to binary {-1, 1} format.
Mathematical requirement for Hinge Loss: max(0, 1 - y_i(w^T x_i + b))
- True (Depressed) → 1
- False (Not Depressed) → -1

In [22]:
# Transform Depression to {-1, 1}
df['Depression'] = df['Depression'].map({True: 1, False: -1})

print("Target variable transformation completed.")
print(f"\nTarget distribution (class balance):")
print(df['Depression'].value_counts())
print(f"\nProportions:")
print(df['Depression'].value_counts(normalize=True).round(4))

Target variable transformation completed.

Target distribution (class balance):
Depression
-1    89938
 1    10062
Name: count, dtype: int64

Proportions:
Depression
-1    0.8994
 1    0.1006
Name: proportion, dtype: float64


## 6. Final Data Export

Save the fully processed dataset to CSV format for use in model training and validation.
Output: `data/processed/clean_student_data.csv`

In [ ]:
import os

# Create output directory if not exists
os.makedirs('data/processed', exist_ok=True)

# Reorder columns: move Depression to the end
cols = [col for col in df.columns if col != 'Depression']
df = df[cols + ['Depression']]

# Save processed data
output_path = 'data/processed/clean_student_data.csv'
df.to_csv(output_path, index=False)

print(f"Dataset successfully exported to: {output_path}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Column order: {list(df.columns)}")
print(f"\nFinal dataset preview:")
df.head()

Dataset successfully exported to: data/processed/clean_student_data.csv

Final dataset shape: (100000, 18)
Final dataset preview:


,Age,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level,Depression,Transportation_Time,Student_Debt,Part_Time_Job,Gender_Male,Department_Business,Department_Engineering,Department_Medical,Department_Science,Living_Status_Famille,Living_Status_Seul
0,0.495403,0.772553,0.262011,-0.612083,-0.069468,0.914222,0.609728,-1,0.480073,0,1,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,-0.504411,-0.228120,-0.909858,1.361535,1.679202,1.559878,-1.496801,-1,-0.666320,0,0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-0.504411,0.305063,-0.974962,-1.118139,-1.145572,1.444582,-0.794625,-1,0.265124,1,0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,-0.004504,1.444970,0.265221,-1.269956,0.737610,1.283168,-0.794625,-1,1.411518,1,0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,-1.004318,0.544076,-0.063508,-0.966322,0.535841,-1.622284,1.311904,-1,-0.021474,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
